In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

MODEL_DIR = "/Users/rohan/Documents/Final year/cwe_project/scripts/codet5_juliet_multiclass"   # folder we saved earlier

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

# Load label → CWE mapping
mapping_df = pd.read_csv("/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv")
cwe_map = dict(zip(mapping_df["cwe_index"], mapping_df["cwe"]))

W1120 13:52:40.571000 71273 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
def predict_cwe(code_snippet):
    inputs = tokenizer(
        code_snippet,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
    
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    pred_index = torch.argmax(probs, dim=-1).item()
    pred_cwe = cwe_map[pred_index]

    return {
        "predicted_index": pred_index,
        "predicted_cwe": pred_cwe,
        "confidence": float(probs[0][pred_index])
    }

In [4]:
test_code = """
void bad() {
    char buf[10];
    gets(buf);  // buffer overflow
}
"""

result = predict_cwe(test_code)
print(result)

{'predicted_index': 7, 'predicted_cwe': 'CWE134', 'confidence': 0.07102970778942108}
